# Stage 1: Non-instruction Causal LLM Fine-tuning or Domain-Adaptive Continued Pretraining

## Pipeline

```text
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

## Continued Pretraining vs Instruction Fine-Tuning

In this notebook, we are performing **continued pretraining / non-instruction fine-tuning** on raw pharma PDF text.

The model is given raw domain text such as:

> Metformin is one of the most widely prescribed oral antihyperglycemic agents...

The model then learns to **predict the next token** from this raw text.

This means the model learns:

- Pharma language
- Drug names
- Medical terminology
- Scientific writing style
- Domain-specific sentence patterns

However, the model is **not explicitly taught**:

- How to answer a user's question
- How to follow instructions
- How to respond in Q&A format
- How to behave like a domain-specific chatbot

---

## What Instruction Fine-Tuning Looks Like

In instruction fine-tuning, the training data is prepared in an **instruction-response format**.

Example:

```json
{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK, which improves glucose uptake and reduces hepatic gluconeogenesis."
}

OR

{
  "messages": [
    {
      "role": "user",
      "content": "What is the primary mechanism of action of Metformin?"
    },
    {
      "role": "assistant",
      "content": "Metformin primarily works by activating AMPK..."
    }
  ]
}

In [1]:
# ============================================================
# 1. Install required libraries
# ============================================================
# PyMuPDF: PDF text extraction
# datasets: Hugging Face dataset creation
# transformers/accelerate: model, tokenizer, Trainer
# peft: LoRA/QLoRA adapters
# bitsandbytes: 4-bit/8-bit quantized loading

!pip install -q -U PyMuPDF transformers accelerate peft bitsandbytes
!pip install -q -U pyarrow
!pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 124.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 36.5 MB/s eta 0:00:00


In [2]:
# ============================================================
# 2. Imports
# ============================================================

import os
import re
import gc
import math
import json
import random
import unicodedata
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import fitz # PyMuPDF
import torch
from datasets import Dataset, DatasetDict

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed
)

from peft import (
    LoraConfig,
    TaskType,
    prepare_model_for_kbit_training,
    get_peft_model,
    PeftModel
)

import warnings
warnings.filterwarnings("ignore")

In [3]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

from dataclasses import dataclass, asdict

@dataclass
class Config:
    # Path of the pharma PDF file that will be used as the raw domain corpus.
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge.pdf"

    # Base causal language model that we will fine-tune on pharma-domain text.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Directory where training checkpoints will be saved during fine-tuning.
    output_dir: str = "/content/pharma_tinyllama_lora_output"

    # Directory where the final trained LoRA adapter will be saved.
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"

    # Directory where cleaned and processed training data will be saved.
    processed_data_dir: str = "/content/pharma_processed_data"

    # Minimum paragraph length required to keep a paragraph for training.
    min_chars_per_paragraph: int = 80

    # Number of tokens in each training block for causal language modeling.
    block_size: int = 512

    # Percentage of data used for validation instead of training.
    test_size: float = 0.15

    # Random seed used to make splitting and training more reproducible.
    seed: int = 42

    # LoRA rank; controls the size and capacity of the trainable adapter.
    lora_r: int = 16

    # LoRA scaling factor; controls the strength of the LoRA update.
    lora_alpha: int = 32

    # Dropout applied inside LoRA layers to reduce overfitting.
    lora_dropout: float = 0.05

    # Number of times the model will see the complete training dataset.
    num_train_epochs: float = 3.0

    # Number of training samples processed per GPU/device at one time.
    per_device_train_batch_size: int = 1

    # Number of validation samples processed per GPU/device at one time.
    per_device_eval_batch_size: int = 1

    # Number of small batches accumulated before one optimizer update.
    gradient_accumulation_steps: int = 8

    # Step size used by the optimizer to update trainable LoRA weights.
    learning_rate: float = 2e-4

    # Fraction of early training steps used to gradually increase learning rate.
    warmup_ratio: float = 0.03

    # Regularization value used to prevent weights from becoming too large.
    weight_decay: float = 0.01

    # Number of training steps after which logs will be printed.
    logging_steps=1
    logging_first_step=True

    # Number of training steps after which validation will be performed.
    eval_steps: int = 10

    # Number of training steps after which a checkpoint will be saved.
    save_steps: int = 25

    # Maximum number of checkpoints to keep; older checkpoints will be deleted.
    save_total_limit: int = 2

    # Maximum number of training steps; -1 means train using num_train_epochs.
    max_steps: int = -1

In [4]:
config = Config()

In [5]:
config

Config(pdf_path='/content/Metformin-Lipid-Therapy-Knowledge.pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_lora_output', adapter_dir='/content/pharma_tinyllama_lora_adapter', processed_data_dir='/content/pharma_processed_data', min_chars_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=3.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, eval_steps=10, save_steps=25, save_total_limit=2, max_steps=-1)

In [6]:
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_lora_adapter",
  "processed_data_dir": "/content/pharma_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


In [7]:
config.model_name

'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'

In [8]:
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [9]:
# ============================================================
# 4. Optional Colab upload helper
# ============================================================
# Run this cell only if your PDF is not already present at config.pdf_path.
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

PDF found: /content/Metformin-Lipid-Therapy-Knowledge.pdf


In [10]:
# # ============================================================
# # 5. Extract text from PDF
# # ============================================================
from typing import List, Dict, Any
import fitz  # PyMuPDF
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages


In [11]:
config.pdf_path

'/content/Metformin-Lipid-Therapy-Knowledge.pdf'

In [12]:
pdf_pages = extract_pdf_pages(config.pdf_path)

In [13]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

Total pages with extracted text: 6
Page-level character counts:
Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters


In [14]:
print(pdf_pages[0]['text'])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

| Cleaning Step                          | Code / Logic                             | What It Does                                                                  | Example Before                                                  | Example After                                                  | Why It Matters for Fine-Tuning                                            |
| -------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------- |
| Unicode normalization                  | `unicodedata.normalize("NFKC", text)`    | Converts unusual Unicode characters into standard readable characters.        | `ＡＭＰＫ`, `ﬁ`                                                     | `AMPK`, `fi`                                                   | Prevents tokenizer confusion caused by hidden or non-standard characters. |
| Remove zero-width characters           | `text.replace("\u200b", "")`             | Removes invisible zero-width spaces from PDF text.                            | `Metformin​ activates AMPK`                                     | `Metformin activates AMPK`                                     | Invisible characters can create bad tokens and noisy training data.       |
| Remove BOM / hidden marker             | `text.replace("\ufeff", "")`             | Removes hidden Byte Order Mark characters sometimes found in extracted text.  | `﻿Metformin is used...`                                         | `Metformin is used...`                                         | Keeps the training text clean and consistent.                             |
| Fix hyphenated line breaks             | `re.sub(r"(\w)-\n(\w)", r"\1\2", text)`  | Joins words that were broken across PDF lines.                                | `gluconeogene-\nsis`                                            | `gluconeogenesis`                                              | Prevents the model from learning broken medical terms.                    |
| Normalize spaces and tabs              | `re.sub(r"[ \t]+", " ", text)`           | Converts multiple spaces or tabs into one space.                              | `Metformin     activates    AMPK`                               | `Metformin activates AMPK`                                     | Makes text consistent and easier for tokenizer/model to learn.            |
| Normalize blank lines                  | `re.sub(r"\n{3,}", "\n\n", text)`        | Converts too many blank lines into a proper paragraph gap.                    | `Para 1\n\n\n\nPara 2`                                          | `Para 1\n\nPara 2`                                             | Preserves paragraph structure without unnecessary whitespace noise.       |
| Remove standalone page numbers         | `re.sub(r"(?m)^\s*\d+\s*$", "", text)`   | Removes lines that contain only page numbers.                                 | `1` or `23`                                                     | Removed                                                        | Prevents the model from learning irrelevant PDF page numbers.             |
| Split into paragraphs                  | `re.split(r"\n\s*\n", text)`             | Splits text wherever there is a blank line.                                   | `Para 1\n\nPara 2`                                              | `["Para 1", "Para 2"]`                                         | Helps preserve meaningful document structure.                             |
| Remove line wrapping inside paragraphs | `re.sub(r"\n+", " ", paragraph)`         | Converts broken lines inside the same paragraph into a single paragraph line. | `Metformin is widely prescribed\noral antihyperglycemic agent.` | `Metformin is widely prescribed oral antihyperglycemic agent.` | Prevents the model from learning artificial PDF line breaks.              |
| Normalize paragraph spacing            | `re.sub(r"\s+", " ", paragraph).strip()` | Removes extra spaces inside each paragraph and trims start/end spaces.        | `  Metformin   activates   AMPK.  `                             | `Metformin activates AMPK.`                                    | Produces clean, readable training examples.                               |
| Remove empty paragraphs                | `if paragraph:`                          | Keeps only non-empty cleaned paragraphs.                                      | `""`                                                            | Removed                                                        | Avoids useless blank samples in the dataset.                              |
| Rebuild cleaned text                   | `"\n\n".join(cleaned_paragraphs)`        | Joins cleaned paragraphs with two newlines.                                   | List of cleaned paragraphs                                      | Clean paragraph-level text                                     | Creates a clean corpus suitable for causal LM training.                   |
| Track cleaned page length              | `char_count: len(cleaned_text)`          | Stores number of characters after cleaning.                                   | Raw page length unknown                                         | `char_count = 1450`                                            | Helps debug whether a page has too little or too much extracted content.  |
| Preview cleaned output                 | `cleaned_pages[0]["text"][:1500]`        | Prints first 1500 characters of cleaned page 1.                               | Full cleaned page                                               | Preview text                                                   | Helps manually verify that cleaning worked correctly.                     |


In [15]:
# ============================================================
# 6. Text cleaning utilities
# ============================================================
import re
import unicodedata

def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" ->  "AMPK"
    # Example: "eﬀiciency" ("ﬀ" is a single ligature character) -> "efficiency"  (split into f + f)
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    # \u200b = zero-width space, \ufeff = byte-order mark
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    # Example: "AMPK    regulates    glucose" -> "AMPK regulates glucose"
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    # Example: "Para1\n\n\n\n\nPara2" -> "Para1\n\nPara2"
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    # Example: "Some text\n  42  \nMore text" -> "Some text\n\nMore text"
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):      # split on blank lines
        paragraph = re.sub(r"\n+", " ", paragraph)    # single newlines inside para → space
        paragraph = re.sub(r"\s+", " ", paragraph).strip()  # clean extra spaces

        if paragraph:  # skip empty ones
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)  # rejoin with clean paragraph breaks

In [16]:
cleaned_pages = []

for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

print("Total cleaned pages:", len(cleaned_pages))

Total cleaned pages: 6


In [17]:
print(cleaned_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.

Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing endogenous cholesterol synthesis

In [18]:
# ============================================================
# 7. Split cleaned pages into paragraphs
# ============================================================
# This step converts cleaned page-level text into paragraph-level records.

def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n\n")

        # enumerate(... start=1) gives us: 1st para, 2nd para, etc
        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [19]:
paragraph_records = split_into_paragraph_records(cleaned_pages)

In [20]:
print("Total paragraph records:", len(paragraph_records))

for record in paragraph_records[:3]:
    print("=" * 80)
    print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
    print(record["text"])

Total paragraph records: 9
Page: 1 | Paragraph: 1 | Characters: 575
Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.
Page: 1 | Paragraph: 2 | Characters: 598
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol abs

In [21]:
# paragraph_records

In [22]:
# ============================================================
# 8. Save extracted and cleaned corpus for auditability
# ============================================================
# In real projects, always save intermediate datasets.
# This helps with reproducibility, debugging, and compliance review.

raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_processed.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

Saved raw pages to: /content/pharma_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraph corpus to: /content/pharma_processed_data/pharma_paragraph_processed.jsonl


In [23]:
# ============================================================
# 9. Create Hugging Face Dataset
# ============================================================
from datasets import Dataset

if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )

# convert list to huggingface dataset format to train the model
text_dataset = Dataset.from_list(paragraph_records)

In [24]:
print(text_dataset)

Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 9
})


In [25]:
print(text_dataset[0])

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.', 'source_page': 1, 'paragraph_id': 1, 'char_count': 575}


In [26]:
# ============================================================
# 10. Train/eval split
# ============================================================
# Even for small demos, keep an evaluation set.
# This gives us validation loss and perplexity.

split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

from datasets import DatasetDict
dataset = DatasetDict(
    {
        "train": split_dataset["train"],
        "test": split_dataset["test"],
    }
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 7
    })
    test: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 2
    })
})


## Load tokenizer

The tokenizer converts text into token IDs.

For causal language modeling, the model learns:

```text
Given previous tokens, predict the next token.
```

This is why we call it **non-instruction causal LM fine-tuning**.

In [27]:
# ============================================================
# 11. Load tokenizer
# ============================================================

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# Llama, Mistral, and many decoder-only models don't come with a pad_token.
# They were trained with eos_token to signal end of sequence,
# but never needed padding because they train on 1 sample at a time.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
# Why this breaks training: When you batch multiple examples together, they must be the same length.
# Shorter ones get padded. If there's no pad_token, HuggingFace throws an error.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tells the tokenizer to add padding at the end, not start.
# Why right for causal LM: Decoder-only models like Llama generate left-to-right.
# If you left-pad, the model sees <pad> <pad> hello and the real text starts at position 2.
# That hurts performance because the model's positional embeddings are now offset. With right-padding,
# the real text always starts at position 0.
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [28]:
tokenizer.eos_token

'</s>'

In [29]:
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


In [30]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [31]:
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)

Tokenizing text corpus:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/2 [00:00<?, ? examples/s]

| Parameter                                       | Meaning                                                                                                                               |
| ----------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------- |
| `tokenize_function`                             | This function converts each text example into token IDs.                                                                              |
| `batched=True`                                  | The function processes multiple rows at once instead of one row at a time. This makes tokenization faster.                            |
| `remove_columns=datasets["train"].column_names` | After tokenization, the original dataset columns are removed. Only tokenized columns such as `input_ids` and `attention_mask` remain. |
| `desc="Tokenizing text corpus"`                 | This message is shown in the progress bar so we can understand that tokenization is currently running.                                |


In [32]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 7
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2
    })
})

# attention_mask

attention_mask = "which tokens the model is allowed to look at"
### The problem it solves
Transformers use self-attention: every token looks at every other token. But padded tokens are fake junk. If the model attends to pads, it learns garbage.

### How it works
1 = real token, attend to it
0 = pad token, ignore it

Example: <br>
text = "Hello"
tokens = ["Hello", PAD, PAD, PAD]

input_ids = [101, 2, 2, 2] # 2 = EOS used as pad
attention_mask = [1, 0, 0, 0] # only "Hello" is real

During attention, PAD tokens get 0 attention weight. The model acts like they don't exist.

## What Does `512` Mean in Text Packing?

In this notebook, `512` means the **sequence length** or **block size** used for causal language model training.

It is **not the embedding size**.

It simply means:

> Each training example will contain 512 tokens.

---

## Example

Suppose the tokenizer converts our pharma text into 1,300 tokens:

```text
[token_1, token_2, token_3, ..., token_1300]?

if we set:

block_size = 512

then the tokens are split like this:

Block 1 = token 1 to token 512
Block 2 = token 513 to token 1024
Remaining tokens = token 1025 to token 1300

Is 512 Padding?

Not exactly.

512 is the target length of each training block.

If we use text packing, we try to fill each block with real tokens, so padding is reduced.

Without packing:

Paragraph 1 = 100 tokens + 412 padding tokens
Paragraph 2 = 200 tokens + 312 padding tokens

With packing:

Block 1 = 512 real tokens
Block 2 = 512 real tokens

So 512 is the fixed token length used to make training efficient.

Is 512 Embedding Size?

No.

Embedding size means the hidden vector dimension of the model.

For example, a model may convert each token into a vector like:

token → 2048-dimensional vector

That 2048 is embedding/hidden size.

But 512 here means:

How many tokens we give to the model at one time

In [33]:
def create_training_blocks(tokenized_examples):
    """
    Core idea: Instead of padding every example to block_size,
    you glue all your tokenized texts together into one giant stream,
    then chop it into perfect token chunks. Way more efficient.

    Normal padding is wasteful:
    Ex1: [1,2,3] + [PAD]*2045 → 99.9% wasted compute
    Ex2: [4,5] + [PAD]*2046 → 99.9% wasted compute

    Packing uses 100% of the context window:
    [1,2,3,4,5,6,7,8,...] → chunk into [block_size] [block_size] [block_size]
    """
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    # If total_tokens = 10 and block_size = 4, then usable_tokens = 8. You'd drop the last 2 tokens.
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    # ------------------------------------------------
    # labels = the ground truth next tokens the model should predict at each position.
    # Why do you need labels?
    # Without labels, there's no loss to compute. Training = compare model prediction vs labels and update weights.
    # No labels → no correct answer → no learning.
    # Why DataCollatorForLanguageModeling is critical: DataCollatorForLanguageModeling(mlm=False) fixes 2 things automatically:
    # 1. Shifts for causal LM: Internally treats input_ids[i+1] as the target for position i.
    # 2. Masks pads: Sets labels[pad_positions] = -100. -100 tells the loss function "ignore this position". so it wont train the model to predict padding.
    # EX: input_ids: [The, cat, PAD, PAD]  # from your .copy()
    # labels:    [cat, PAD, -100, -100] # after DataCollator processes it
    # Now the model only learns The→cat. The pads are ignored.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

In [34]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/7 [00:00<?, ? examples/s]

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/2 [00:00<?, ? examples/s]

In [35]:
final_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 0
    })
})

In [36]:
sample = final_dataset["train"][0]

In [38]:
sample.items()

dict_items([('input_ids', [1, 1963, 22824, 28460, 26101, 3630, 448, 9305, 29871, 29945, 9305, 29871, 29945, 448, 319, 29902, 297, 360, 11124, 8565, 22205, 322, 1963, 22824, 346, 329, 936, 390, 29987, 29928, 29936, 1963, 22824, 29899, 7247, 1034, 13364, 6081, 363, 2888, 2691, 29899, 29873, 27964, 322, 390, 10051, 7639, 362, 29889, 7519, 29883, 1288, 2793, 871, 29936, 451, 16083, 9848, 29889, 17157, 29769, 3012, 928, 616, 21082, 338, 10231, 368, 1304, 297, 1374, 22824, 346, 329, 936, 5925, 304, 27599, 20853, 1199, 29892, 1301, 924, 290, 1199, 29892, 3279, 290, 1199, 29892, 17135, 17292, 327, 7384, 29892, 22233, 9562, 29892, 322, 24899, 936, 20035, 29889, 512, 3646, 29769, 29892, 4933, 6509, 4733, 508, 7536, 277, 675, 2531, 267, 470, 3279, 1144, 393, 1122, 1708, 3269, 284, 16178, 297, 17135, 4768, 3002, 29889, 4525, 27303, 526, 9324, 6419, 746, 23387, 411, 17986, 8845, 29892, 2224, 1582, 7418, 29892, 5199, 2531, 300, 1199, 29892, 322, 17135, 29899, 276, 6591, 4768, 290, 935, 414, 29889, 3

In [37]:
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 512
labels length: 512
Decoded sample preview:

<s> Pharma Domain Training Data - Page 5 Page 5 - AI in Drug Discovery and Pharmaceutical R&D; Pharma-domain corpus extension for custom fine-tuning and RAG experimentation. Educational content only; not medical advice. Target identification Artificial intelligence is increasingly used in pharmaceutical research to analyze genomics, transcriptomics, proteomics, disease phenotypes, chemical libraries, and clinical datasets. In target identification, machine learning models can prioritize genes or proteins that may play causal roles in disease biology. These predictions are strengthened when integrated with experimental validation, pathway analysis, human genetics, and disease-relevant biomarkers. Molecular screening In early discovery, deep learning can support virtual screening by predicting protein-ligand binding affinity, molecular properties, toxicity signals,

## Load Model for QLoRA Training

In this step, we load the base model for fine-tuning.

If GPU is available, we load the model in **4-bit mode**.

This helps because:

- It uses less GPU memory
- It allows us to fine-tune larger models on limited hardware
- It is useful for Colab or small GPU environments
- It works well with LoRA/QLoRA fine-tuning

If GPU is not available, the model will load normally on CPU, but training will be much slower.

In [38]:
# ============================================================
# 13. Load base model
# ============================================================
import torch
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [39]:
# Clear memory before loading the model.
import gc
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [40]:
from transformers import AutoModelForCausalLM

# Only import quantization stuff if you have a GPU. bitsandbytes doesn't work on CPU.
if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,                    # Use 4-bit instead of 16/32-bit
        bnb_4bit_quant_type="nf4",            # NormalFloat4: best 4-bit format for LLMs
        bnb_4bit_compute_dtype=torch.float16, # Do math in fp16 even though weights are 4-bit
        # (4-bit is only for storage.
        # GPUs can't do matrix math in 4-bit. You have to "dequantize" to fp16/bf16 to actually compute.
        # GPUs have fp16/bf32/INT8 tensor cores, but no 4-bit tensor cores yet. )
        bnb_4bit_use_double_quant=True,       # Quantize the quantization constants too
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",        # Spread model across GPUs automatically
        trust_remote_code=True,   # Needed for Llama, Qwen, etc that have custom code
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,  # No quantization on CPU
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Base model loaded successfully.


In [41]:
# ============================================================
# 14. Apply LoRA adapters
# ============================================================
# LoRA trains a small number of adapter parameters instead of updating all base model weights.
# This is cheaper than full fine-tuning and is widely used in real projects.
from peft import LoraConfig
from peft import TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,     # Tells PEFT this is next-token prediction
    r=config.lora_r,                  # Rank of the adapter
    lora_alpha=config.lora_alpha,     # Scaling factor
    lora_dropout=config.lora_dropout, # Dropout on LoRA layers
    bias="none",                      # Don't train bias terms
    target_modules=[                  # Which layers to inject LoRA into
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [42]:
from peft import get_peft_model
model = get_peft_model(base_model, lora_config)

In [43]:
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [44]:
# ============================================================
# 15. Data collator
# ============================================================
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Why Do We Need `DataCollatorForLanguageModeling`?

After tokenization and text packing, our dataset contains token IDs in a training-ready structure.

However, the `Trainer` still needs a component that can take multiple examples from the dataset and convert them into a proper batch during training.

That component is called a **data collator**.

```python
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
What Does the Data Collator Do?

The data collator prepares mini-batches for the model.

It handles things like:

Collecting multiple training examples together
Padding sequences if required
Converting examples into tensors
Preparing labels for language modeling
Example

Suppose our packed dataset has training examples like this:

Example 1 = 512 tokens
Example 2 = 512 tokens
Example 3 = 512 tokens

During training, the Trainer may take two examples at a time:

Batch = Example 1 + Example 2

The data collator converts them into tensors like:

input_ids shape      = [2, 512]
attention_mask shape = [2, 512]
labels shape         = [2, 512]

This is the format the model expects during training.

Why mlm=False?

mlm means Masked Language Modeling.

Masked Language Modeling is used for BERT-style models.

Example:

Metformin is used for [MASK].

The model predicts the masked word:

diabetes

But we are using TinyLlama, which is a causal language model.

Causal language models learn by predicting the next token from left to right.

Example:

Metformin → is
Metformin is → used
Metformin is used → for
Metformin is used for → diabetes

So we set:

mlm=False

This tells Hugging Face:

Do not use BERT-style masked language modeling. Use causal language modeling instead.

Why Is This Needed Even After Tokenization and Packing?

Tokenization converts text into token IDs.

Text packing groups token IDs into fixed-size blocks.

But the data collator prepares those blocks into actual training batches.

So the flow is:

Raw pharma text
   ↓
Tokenization
   ↓
Token IDs
   ↓
Text packing
   ↓
Fixed-size training blocks
   ↓
Data collator
   ↓
Mini-batches for Trainer
   ↓
Model training

In [45]:
# ============================================================
# 16. Training arguments
# ============================================================
# These settings are designed for a demo run.
# For larger corpora, increase dataset size, epochs, and evaluation frequency carefully.

from transformers import TrainingArguments

training_kwargs = dict(
    output_dir=config.output_dir,   # Where to save checkpoints + final model
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size, # Eval batch size
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,          # Print loss every N steps
    logging_first_step=True,  # Also log step 0

    eval_steps=config.eval_steps, # Run eval every N steps
    save_steps=config.save_steps, # Save checkpoint every N steps
    save_total_limit=config.save_total_limit, # Keep only last N checkpoints
    fp16=use_cuda,# Use fp16 mixed precision on GPU
    bf16=False,   # Use bf16 instead of fp16
    report_to="none", # Don't send logs to WandB/MLflow
    remove_unused_columns=False, # Keep all dataset columns
)

training_args = TrainingArguments(**training_kwargs)

In [46]:
# ============================================================
# 17. Build Trainer
# ============================================================
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["test"],
    data_collator=data_collator,
)
print("Trainer is ready.")

Trainer is ready.


In [47]:
# ============================================================
# 18. Start training
# ============================================================
train_result = trainer.train()
print("Training completed.")

Step,Training Loss
1,2.148971
2,2.148971
3,2.124230


Training completed.


In [48]:
for log in trainer.state.log_history:
    print(log)

{'loss': 2.14897084236145, 'grad_norm': 0.6705529093742371, 'learning_rate': 0.0, 'epoch': 1.0, 'step': 1}
{'loss': 2.1489710807800293, 'grad_norm': 0.6784319281578064, 'learning_rate': 4e-05, 'epoch': 2.0, 'step': 2}
{'loss': 2.124229669570923, 'grad_norm': 0.6561076641082764, 'learning_rate': 8e-05, 'epoch': 3.0, 'step': 3}
{'train_runtime': 21.198, 'train_samples_per_second': 0.849, 'train_steps_per_second': 0.142, 'total_flos': 57901993426944.0, 'train_loss': 2.1407238642374673, 'epoch': 3.0, 'step': 3}


In [49]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

LoRA adapter saved to: /content/pharma_tinyllama_lora_adapter
Saved files:
['README.md', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json']


In [50]:
from huggingface_hub import notebook_login

# Log in to Hugging Face Hub. A popup will appear to enter your token.
# You can find your token at https://huggingface.co/settings/tokens
notebook_login()

In [53]:
# ============================================================
# Push Stage 1 non-instruction LoRA adapter to Hugging Face
# ============================================================

from huggingface_hub import HfApi, create_repo
from pathlib import Path

def push_to_hub(
    repo_id: str,
    kind: str,                  # "dataset" or "model"
    local_path=None,            # file path (dataset) or dir path (model/adapter/tokenizer)
    path_in_repo: str = "",     # subfolder/filename inside the repo (e.g. "instruction/data.jsonl" or "stage1_adapter")
    private: bool = False,
):
    """
    Push a dataset file or a model/adapter/tokenizer folder to the HF Hub.

    Examples:
        # Dataset file -> subfolder in a dataset repo
        push_to_hub(
            "SivaSai8143/pharma-finetuning-data", "dataset",
            local_path="/content/pharma_instruction_dataset.jsonl",
            path_in_repo="instruction/pharma_instruction_dataset.jsonl",
        )

        # Adapter + tokenizer folder -> model repo
        push_to_hub(
            "SivaSai8143/pharma-tinyllama-instruction-lora-adapter", "model",
            local_path="/content/pharma_tinyllama_lora_adapter",
        )

        # Merged model folder -> model repo
        push_to_hub(
            "SivaSai8143/pharma-tinyllama-instruction-merged", "model",
            local_path="/content/pharma_tinyllama_merged",
        )
    """
    repo_type = "dataset" if kind == "dataset" else "model"
    api = HfApi()
    create_repo(repo_id=repo_id, repo_type=repo_type, private=private, exist_ok=True)

    local_path = Path(local_path)

    if local_path.is_dir():
        api.upload_folder(
            folder_path=str(local_path),
            path_in_repo=path_in_repo,
            repo_id=repo_id,
            repo_type=repo_type,
        )
    else:
        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=path_in_repo or local_path.name,
            repo_id=repo_id,
            repo_type=repo_type,
        )

    print(f"Pushed {local_path} -> {repo_id} ({repo_type})")

# Dataset file -> subfolder in a dataset repo
push_to_hub("SivaSai8143/pharma-finetuning-data", "dataset",
    local_path="/content/pharma_processed_data/pharma_paragraph_processed.jsonl",
    path_in_repo="raw/pharma_paragraph_process.jsonl"
)

# Adapter + tokenizer folder -> model repo
push_to_hub(
    "SivaSai8143/pharma-tinyllama-non-instruction-lora-adapter", "model",
    local_path="/content/pharma_tinyllama_lora_adapter",
)

In [51]:
# ============================================================
# 21. Reload base model + LoRA adapter correctly
# ============================================================
# Clean old objects to free memory.

del trainer

try:
    del model
    del base_model
except NameError:
    pass

gc.collect()

if use_cuda:
    torch.cuda.empty_cache()

In [52]:
from transformers import AutoTokenizer
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [53]:
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [54]:
from peft import PeftModel
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)

print("Base model + LoRA adapter loaded successfully for inference.")

Base model + LoRA adapter loaded successfully for inference.


In [55]:
inference_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [56]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations,
# not chat-style questions.

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [57]:
# ============================================================
# 23. Test text continuation
# ============================================================
# These prompts are continuation-style prompts.
# In Notebook 2, we will create instruction prompts for Q&A.

prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]


In [58]:
# ============================================================
# 23. Test text continuation
# ============================================================

for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

PROMPT:
Metformin is one of the most widely prescribed oral antihyperglycemic agents

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Metformin is one of the most widely prescribed oral antihyperglycemic agents in the world. It is a sulfonylurea class drug, which has been used to treat type 2 diabetes for over 30 years. However, despite its proven efficacy and safety, the mechanism by which it works remains unknown. A recent study from the National Institutes of Health (NIH) reveals that Metformin can directly affect the cellular processes that regulate the production of insulin. This finding could lead to the development of new drugs targeting these processes.
"These findings are very important because they suggest that Met

PROMPT:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe, a statin drug that reduces cholesterol and LDL cholesterol, lowers LDL-C by 39% compared to statins alone.
The study was published online in the New England Journal of Medicine.
Atorvastatin is a lipid (fat) lowering drug used for primary prevention of cardiovascular disease, as well as secondary prevention of coronary artery disease, strokes and other types of heart attacks.
Atrial fibrillation is an irregular heartbeat that causes unpredict

PROMPT:
Artificial intelligence is transforming pharmaceutical research by accelerating

MODEL CONTINUATION:
Artificial intelligence is transforming pharmaceutical research by accelerating drug discovery, reducing time-to-market and increasing the efficiency of R&D operations.
Industrial IoT is driving the next wave of innovation in healthcare as a growing number of devices and apps are connected to create smart environments for better patient care.
MedTech IoT is revol

In [59]:
# ============================================================
# 24. Optional merge step
# ============================================================
# This step merges the LoRA adapter into the base model.
# Use this only when you want a standalone model for deployment.

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

import os
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

merged_model_dir = "/content/pharma_tinyllama_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)


# Reload the base model in float16 for safe merging.
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    config.adapter_dir
)

# Merge LoRA adapter weights into the base model weights.
merged_model = model_with_adapter.merge_and_unload()

# Save the merged standalone model and tokenizer.
merged_model.save_pretrained(merged_model_dir)
inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/pharma_tinyllama_merged_model


In [68]:
# ============================================================
# Push Stage 1 non-instruction Merged model to Hugging Face
# ============================================================

# Merged model folder -> model repo
push_to_hub(
    "SivaSai8143/pharma-tinyllama-non-instruction-merged", "model",
    local_path="/content/pharma_tinyllama_merged_model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...d_model/model.safetensors:   1%|          | 16.0MB / 2.20GB            

Pushed /content/pharma_tinyllama_merged_model -> SivaSai8143/pharma-tinyllama-instruction-merged (model)


# Stage 2: Instruction Fine-Tuning on the Same Domain-Adapted Finetuned Model

In Stage 1, we performed **non-instruction fine-tuning / domain-adaptive continued pretraining** on raw pharma PDF text.

Now we continue from the **same Stage 1 LoRA adapter** and perform **instruction fine-tuning** using structured (Alpaca format) pharma instruction-response examples.

```text
Base TinyLlama
   ↓
Stage 1: Raw pharma text continued pretraining using LoRA
   ↓
Stage 1 domain-adapted LoRA adapter
   ↓
Stage 2: Instruction fine-tuning on pharma Q&A data
   ↓
Final instruction-tuned pharma LoRA adapter
```

This means we are not starting from scratch. We are continuing from the model adapter trained in the previous stage.

What changes in instruction fine-tuning?

For non-instruction fine-tuning, the data looked like raw text:

```text
Metformin is one of the most widely prescribed oral antihyperglycemic agents...
```

For instruction fine-tuning, the data looks like below (Alpaca format):

```json
{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK..."
}
```

This teaches the model not only pharma language, but also how to answer user instructions.

In [60]:
instruction_data_path = "/content/pharma_instruction_dataset.jsonl"

In [61]:
from datasets import load_dataset

instruction_dataset = load_dataset(
    "json", # Format: json or jsonl
    data_files=instruction_data_path,
    split="train" # Treat whole file as train split
)

Generating train split: 0 examples [00:00, ? examples/s]

In [62]:
print(instruction_dataset)

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic'],
    num_rows: 48
})


In [63]:
print(instruction_dataset[0])

{'instruction': 'Explain the primary mechanism of action of metformin.', 'input': '', 'output': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


In [64]:
# ============================================================
# Format instruction records
# ============================================================
# This function converts your structured instruction/input/output json into a single text string in Alpaca format.
# The model learns to complete text, so you need to flatten everything into one prompt.

def format_instruction_record(record):
    instruction = str(record.get("instruction", "")).strip()
    input_text = str(record.get("input", "")).strip()
    output_text = str(record.get("output", "")).strip()

    if input_text:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output_text}"
        )
    else:
        text = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{output_text}"
        )

    return {"text": text}

instruction_dataset = instruction_dataset.map(format_instruction_record)

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

In [65]:
instruction_dataset

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
    num_rows: 48
})

In [66]:
print(instruction_dataset[0]["text"])

### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.


In [67]:
# ============================================================
# Create train-validation split
# ============================================================
# This splits single dataset into 80% train + 15% validation. You need both for Trainer.

instruction_datasets = instruction_dataset.train_test_split(
    test_size=0.15,  # 15% for eval, 85% for train
    seed=42          # Reproducible shuffle
)

instruction_datasets["validation"] = instruction_datasets.pop("test")

print(instruction_datasets)
print("Train examples:", len(instruction_datasets["train"]))
print("Validation examples:", len(instruction_datasets["validation"]))

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'source_page', 'topic', 'text'],
        num_rows: 8
    })
})
Train examples: 40
Validation examples: 8


In [68]:
# ============================================================
# Tokenize instruction dataset
# ============================================================
"""
Initialize tokenizer and configure padding for causal LM training.

Key steps:
1. Load the fast tokenizer tied to the base model in config.model_name.
2. Set pad_token = eos_token if missing. Llama/TinyLlama have no PAD by default.
   Causal LM doesn't attend to PADs, so reusing EOS is safe and avoids resizing embeddings.
3. Prints pad_token for sanity check before training.

Why this matters:
- DataCollatorForLanguageModeling needs tokenizer.pad_token_id to pad batches.
- Without it, collation crashes with "Cannot pad non-existent pad token".
- Using eos_token prevents adding new embeddings, keeping LoRA params low.
"""

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.model_name,
    use_fast=True # Rust-based, ~10x faster for.map()
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(tokenizer.pad_token)

</s>


In [69]:
def tokenize_instruction_function(examples):
    """
    Tokenize Alpaca-format instruction text for causal LM training.

    Converts raw "text" field into input_ids, attention_mask, and labels.
    Uses static padding to instruction_max_length so every example has equal length.

    Args:
        examples (dict): Batch from Dataset with key "text".
            Example: {"text": ["### Instruction:\n...\n\n### Response:\n...",...]}

    Returns:
        dict: Tokenized batch containing:
            - input_ids (List[List[int]]): Token IDs, padded to max_length.
            - attention_mask (List[List[int]]): 1 for real tokens, 0 for PAD.
            - labels (List[List[int]]): Shifted input_ids with PAD masked to -100.
              -100 is ignored by CrossEntropyLoss, so model doesn't learn PAD.

    Notes:
        1. Padding="max_length": All sequences become 512 tokens.
           NOTE: For efficiency use padding=False and let
           DataCollatorForLanguageModeling pad per batch instead.
        2. Labels = input_ids: Standard for causal LM. Model predicts next token.
        3. PAD tokens in labels set to -100: Prevents loss on padding positions.
        4. If you later use DataCollatorForLanguageModeling, it will overwrite
           "labels" anyway. This manual version is only needed with
           default_data_collator or custom collators.
    """
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )

    # For causal LM, labels are copied from input_ids.
    tokens["labels"] = tokens["input_ids"].copy()

    # Mask PAD tokens so they don't contribute to loss.
    # attention_mask=0 means PAD -> set label to -100
    tokens["labels"] = [
        [
            token if mask == 1 else -100
            for token, mask in zip(input_ids, attention_mask)
        ]
        for input_ids, attention_mask in zip(tokens["input_ids"], tokens["attention_mask"])
    ]

    return tokens

When we tokenize instruction data, all examples are not the same length.

Example:

Example 1 = 20 tokens
Example 2 = 80 tokens
Example 3 = 150 tokens

But for training, we often make every example the same length, like:

max_length = 512

So shorter examples get extra padding tokens.

Example:

Real text tokens + padding tokens = 512 tokens

Now the problem is:

We want the model to learn from real text, not from padding.

So we use -100 in labels.

-100 tells PyTorch:

Ignore this position while calculating loss.

In [70]:
"""
Tokenize the instruction dataset and drop raw text columns.

Operation details:
1. batched=True: tokenize_instruction_function receives dict of lists,
   e.g. {"text": [str1, str2, ...]}. Faster than single-row calls.
2. remove_columns=instruction_datasets["train"].column_names: Drops original
   columns like ["instruction", "input", "output", "text"]. After this step
   the dataset only contains ["input_ids", "attention_mask", "labels"].
   Trainer only needs tensors, keeping text wastes RAM and disk.
3. desc: Progress bar label shown during .map().

Applied to both splits:
    instruction_datasets["train"] -> instruction_tokenized_datasets["train"]
    instruction_datasets["validation"] -> instruction_tokenized_datasets["validation"]

Why remove_columns:
- Original text can be 10-100x larger than tokens. Deleting it prevents OOM.
- Trainer will error if you pass string columns it can't collate.

"""

instruction_tokenized_datasets = instruction_datasets.map(
    tokenize_instruction_function,
    batched=True,
    remove_columns=instruction_datasets["train"].column_names,
    desc="Tokenizing instruction dataset",
)

print(instruction_tokenized_datasets)

Tokenizing instruction dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing instruction dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 8
    })
})


| Point                          | Approach 1: Continue Same Stage 1 LoRA Adapter                                         | Approach 2: Merge Stage 1, Then Add New LoRA Adapter                                                      |
| ------------------------------ | -------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------- |
| Flow                           | Base model + Stage 1 LoRA adapter → continue training same adapter on instruction data | Base model + Stage 1 LoRA adapter → merge → load merged model → add new LoRA adapter for instruction data |
| Main idea                      | The same adapter learns both domain language and instruction-following behavior        | Stage 1 knowledge becomes part of the merged model, then a new adapter learns instruction behavior        |                                                              |
| Merge required before Stage 2? | No                                                                                     | Yes                                                                                                       |
| Risk/complexity                | Lower complexity                                                                       | Higher complexity, especially if Stage 1 model was loaded in 4-bit/QLoRA mode                             |
| Output size                    | Small final LoRA adapter                                                               | Large merged Stage 1 model + small Stage 2 adapter                                                        |
| Hugging Face upload            | Easy, only adapter can be pushed                                                       | Heavier because merged model is large                                                                     |
| Best for demo           | Best choice                                                                            | Use only if you already merged Stage 1                                                                    |
| Best for production            | Good for experimentation and adapter-based deployment                                  | Useful when you want Stage 1 knowledge permanently inside the base model                                  |
| Recommended for your notebook? | **Yes, recommended**                                                                   | Only if Stage 1 adapter is already merged                                                                 |


In [71]:
# ============================================================
# Load merged Stage 1 model and add new LoRA adapter for instruction tuning
# ============================================================
"""
Stage-2 setup: Take the merged Stage-1 model and wrap it with a new LoRA adapter.

Pipeline:
    Base TinyLlama --Stage1 LoRA--> Merge --> pharma_tinyllama_merged_model
                                        |
                                        v
    Merged Model --Stage2 LoRA--> Instruction-tuned model
"""

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# Clear Python + CUDA cache before loading big model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

use_cuda = torch.cuda.is_available()

merged_model_dir = "/content/pharma_tinyllama_merged_model"

if use_cuda:
    # Load merged Stage 1 model in 4-bit mode for QLoRA instruction tuning.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        # 4-bit quantization config for QLoRA Stage-2
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, # Load base weights in 4-bit NF4
            bnb_4bit_quant_type="nf4", # Best for LLMs
            bnb_4bit_compute_dtype=torch.float16, # Compute in bf16
            bnb_4bit_use_double_quant=True, # Double quant saves 0.4 bits/param
        ),
        device_map="auto", # Shard across GPUs
        trust_remote_code=True,
    )

    instruction_base_model = prepare_model_for_kbit_training(instruction_base_model)

else:
    # CPU fallback. Training on CPU will be slow.
    instruction_base_model = AutoModelForCausalLM.from_pretrained(
        merged_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

instruction_base_model.config.use_cache = False

# Create a new LoRA adapter for instruction fine-tuning.
instruction_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

instruction_model = get_peft_model(
    instruction_base_model,
    instruction_lora_config
)

instruction_model.print_trainable_parameters()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [72]:

# ============================================================
# Instruction fine-tuning data collator
# ============================================================
"""
DataCollatorForLanguageModeling for causal LM instruction tuning.

What it does on each training batch:
1. Pads input_ids and attention_mask to the longest sequence in the batch.
   Uses tokenizer.pad_token_id. This is "dynamic padding" = faster than
   padding all examples to max_length upfront.
2. Creates labels by copying input_ids, then sets label = -100 anywhere
   input_ids == pad_token_id. Loss ignores -100, so model won't learn PAD.
3. mlm=False: Disables masked language modeling. For causal LM we predict
   next token, not masked tokens like BERT.

When to use this vs manual labels:
- If you did padding=False in tokenize_function and did NOT create labels,
  use this collator. It handles everything.
- If you did padding="max_length" and already created labels with -100,
  you can use default_data_collator instead. This collator would overwrite
  your labels anyway.

Performance note:
Dynamic padding saves 5-10x memory. Example: batch of 8 sequences with
lengths [20, 25, 30, 512] gets padded to 512, not all to 512. With static
padding you'd waste more tokens per batch.

Args:
    tokenizer: Your tokenizer, used for pad_token_id.
    mlm=False: Causal LM mode. Set True only for BERT-style models.
"""
from transformers import DataCollatorForLanguageModeling

instruction_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False, # Causal LM: predict next token, not masked tokens
)

In [73]:
# ============================================================
# Instruction fine-tuning arguments
# ============================================================
"""
Define output directories for Stage-2 instruction tuning artifacts.

Paths:
1. instruction_output_dir: Trainer checkpoints, logs, training_args.json.
   Contains full snapshots every save_steps including optimizer states.
   Can be 5-20GB. Safe to delete after training if you only need final adapter.

2. instruction_adapter_dir: Final LoRA adapter only.
   Contains adapter_config.json + adapter_model.bin (~60MB for r=16).
   This is what you load for inference or merge into base model.

"""
instruction_output_dir = "/content/pharma_tinyllama_instruction_lora_output"
instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"

os.makedirs(instruction_output_dir, exist_ok=True)
os.makedirs(instruction_adapter_dir, exist_ok=True)

In [74]:
from transformers import TrainingArguments

instruction_training_args = TrainingArguments(
    output_dir=instruction_output_dir,
    # === Duration ===
    num_train_epochs=5, # Ignored because max_steps is set
    max_steps=5, # Stop after 5 optimizer updates. Use for quick test.
                  # Delete this line for full training and use epochs instead.

    # === Batch & Memory ===
    per_device_train_batch_size=1, # 1 sample per GPU. Low VRAM usage.
    per_device_eval_batch_size=1, # Eval batch size. Match train for simplicity.
    gradient_accumulation_steps=8, # Accumulate grads over 8 fwd passes.
                                   # Simulates batch_size=8 without OOM.
                                   # 1 * 8 = effective batch 8.

    # === Optimizer ===
    learning_rate=1e-4, # Standard for LoRA. Full finetune uses 2e-5.
    warmup_steps=2, # LR linearly increases 0 -> 1e-4 over first 2 steps.
                     # Prevents early gradient spikes. Rule: 3-10% of total steps.
    weight_decay=0.01, # L2 regularization. Prevents overfitting.

    # === Logging ===
    logging_steps=1, # Print loss every step. Set 10-50 for real runs.
    logging_first_step=True, # Log step 0 to verify setup.

    # === Evaluation ===
    eval_strategy="steps", # Run eval every eval_steps
    eval_steps=1, # Eval every step. Expensive. Use 50-200 normally.
                   # With max_steps=5 this runs 5 evals total.

    # === Checkpointing ===
    save_steps=5, # Save checkpoint every 5 steps. Matches max_steps.
    save_total_limit=2, # Keep only last 2 checkpoints. Deletes older ones.
                        # Saves disk: each checkpoint = 3-5GB with optimizer.

    # === Mixed Precision ===
    fp16=use_cuda, # Use fp16 if GPU available. Faster, less VRAM.
    bf16=False, # Use bf16 on A100/3090+. More stable than fp16.
                 # Don't set both True. Pick one based on GPU.

    # === Misc ===
    report_to="none", # Disable external logging tools.
    remove_unused_columns=False, # Keep required columns.
)

print(instruction_training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=1,
eval_strategy=IntervalStrategy.STEPS,
eval_us

In [75]:
# ============================================================
# Build instruction Trainer
# ============================================================
"""
Construct Hugging Face Trainer for Stage-2 instruction tuning.

The Trainer wraps model, data, and training loop. It handles:
1. Batching: Uses data_collator to pad sequences and create labels.
2. Forward/backward: Computes loss, calls loss.backward().
3. Optimizer step: Updates only LoRA weights every gradient_accumulation_steps.
4. Evaluation: Runs model.eval() on eval_dataset every eval_steps.
5. Checkpointing: Saves model + optimizer + scheduler to output_dir.
"""

from transformers import Trainer

instruction_trainer = Trainer(
    model=instruction_model, # Your PEFT model with Stage-2 LoRA
    args=instruction_training_args, # lr, batch, steps, fp16, etc
    train_dataset=instruction_tokenized_datasets["train"], # 85% of data
    eval_dataset=instruction_tokenized_datasets["validation"], # 15% of data
    data_collator=instruction_data_collator, # Dynamic padding + label creation
)

print("Instruction Trainer is ready.")

Instruction Trainer is ready.


In [76]:
# ============================================================
# Start instruction fine-tuning
# ============================================================

instruction_train_result = instruction_trainer.train()

print("Instruction fine-tuning completed.")
print(instruction_train_result)

Step,Training Loss,Validation Loss
1,2.059745,2.300103
2,2.173541,2.255160
3,2.479143,2.168474
4,2.056589,2.113118
5,2.104938,2.086316


Instruction fine-tuning completed.
TrainOutput(global_step=5, training_loss=2.1747912406921386, metrics={'train_runtime': 40.2564, 'train_samples_per_second': 0.994, 'train_steps_per_second': 0.124, 'total_flos': 128671096504320.0, 'train_loss': 2.1747912406921386, 'epoch': 1.0})


In [77]:
# ============================================================
# Save final instruction-tuned LoRA adapter
# ============================================================
# Persist the Stage-2 LoRA adapter and tokenizer for inference or merging.
# This adapter now contains Stage 1 domain adaptation + Stage 2 instruction tuning.

import os

instruction_adapter_dir = "/content/pharma_tinyllama_instruction_lora_adapter"
os.makedirs(instruction_adapter_dir, exist_ok=True)

# Save only LoRA adapter weights + config.
instruction_trainer.model.save_pretrained(instruction_adapter_dir)
# Save tokenizer to same dir for easy packaging
tokenizer.save_pretrained(instruction_adapter_dir)

print(f"Final instruction-tuned LoRA adapter saved to: {instruction_adapter_dir}")
print(os.listdir(instruction_adapter_dir))

Final instruction-tuned LoRA adapter saved to: /content/pharma_tinyllama_instruction_lora_adapter
['README.md', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json']


In [1]:
# ============================================================
# Push Stage 2 instruction LoRA adapter to Hugging Face
# ============================================================

# # Dataset file -> subfolder in a dataset repo
# push_to_hub("SivaSai8143/pharma-finetuning-data", "dataset",
#     local_path="/content/pharma_instruction_dataset.jsonl",
#     path_in_repo="instruction/pharma_instruction_dataset.jsonl"
# )

# # Adapter + tokenizer folder -> model repo
# push_to_hub(
#     "SivaSai8143/pharma-tinyllama-instruction-lora-adapter", "model",
#     local_path='/content/pharma_tinyllama_instruction_lora_adapter',

# )

In [85]:
import os

card_path = "/content/pharma_tinyllama_instruction_lora_adapter/README.md"

# Only fix if the card exists
if os.path.exists(card_path):
    with open(card_path, "r") as f:
        content = f.read()

    content = content.replace(
        "base_model: /content/pharma_tinyllama_merged_model",
        "base_model: SivaSai8143/pharma-tinyllama-non-instruction-merged"
    )

    with open(card_path, "w") as f:
        f.write(content)
    print("Fixed base_model in README.md")

# Now push
push_to_hub(
    "SivaSai8143/pharma-tinyllama-instruction-lora-adapter", "model",
    local_path="/content/pharma_tinyllama_instruction_lora_adapter",
)

Fixed base_model in README.md


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

Pushed /content/pharma_tinyllama_instruction_lora_adapter -> SivaSai8143/pharma-tinyllama-instruction-lora-adapter (model)


In [78]:
# ============================================================
# Reload final instruction-tuned adapter for inference
# ============================================================

# Load merged Stage-1 base + Stage-2 LoRA adapter for inference.
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
         merged_model_dir, # Contains Stage-1 merged weights
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Attach Stage-2 LoRA on top of Stage-1 merged base
final_instruction_model = PeftModel.from_pretrained(
    base_model,
    instruction_adapter_dir,
)

final_instruction_model.eval()

print("Final instruction-tuned model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Final instruction-tuned model loaded successfully.


In [79]:
# ============================================================
# Instruction-style inference helper
# ============================================================

"""
Format instruction-tuning prompts to match Alpaca-style template.

Template must be identical to training data format. If training used
"### Instruction:\n...\n\n### Response:\n", inference must use the same
or the model won't know where to start generating.

Args:
    instruction (str): The task description. Required.
        Example: "Summarize the following drug interaction."
    input_text (str, optional): Context/data for the task. Default "".
        Example: "Drug A: Warfarin. Drug B: Aspirin."
        If empty, uses 2-section format. If provided, uses 3-section format.

Returns:
    str: Formatted prompt ending with "### Response:\n". Model will
         continue generating tokens after this marker.

Format logic:
1. Two-section format: Used when no input_text. Model sees only instruction.
   ### Instruction:\n{instruction}\n\n### Response:\n

2. Three-section format: Used when input_text provided. Model sees instruction
   + context, then generates response.
   ### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n

"""
def build_instruction_prompt(instruction, input_text=""):
    instruction = instruction.strip() # Remove whitespace to match training
    input_text = input_text.strip() # Prevent empty lines if input_text="  "

    if input_text: # 3-section format: Instruction + Input + Response
        return (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n" # Model generates from here
        )

    # 2-section format: Instruction + Response only
    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n" # Model generates from here
    )


In [80]:
def generate_instruction_response(instruction, input_text="", max_new_tokens=150):
    """
    Generate a response from the instruction-tuned model using Alpaca prompt format.

    Flow:
    1. build_instruction_prompt(): Formats input as "### Instruction:\n...\n\n### Response:\n"
       Must match training template exactly or model won't follow format.
    2. tokenizer(): Converts prompt string -> input_ids + attention_mask tensors.
       return_tensors="pt" gives PyTorch tensors on CPU.
    3..to(device): Moves tensors to same device as model (cuda:0 or cpu).
       Prevents "Expected all tensors to be on same device" error.
    4. torch.no_grad(): Disables gradient tracking. Saves memory, speeds up inference.
       No backward pass needed during generation.
    5. model.generate(): Autoregressively samples tokens until eos_token or max_new_tokens.

    Args:
        instruction (str): Task description. Example: "Explain the side effects".
        input_text (str, optional): Context for task. Default "". If empty, uses
            2-section prompt. If provided, uses 3-section with ### Input: block.
        max_new_tokens (int): Max tokens to generate after prompt. Default 150.
            Total length = len(prompt_tokens) + max_new_tokens. Cap at 512 for
            TinyLlama to avoid exceeding context window.

    Generation params:
        do_sample=True: Use sampling vs greedy decode. Enables temperature/top_p.
        temperature=0.7: Controls randomness. 0.0=deterministic, 1.0=high entropy.
            0.7 is balanced for instruction tasks.
        top_p=0.9: Nucleus sampling. Keep tokens comprising 90% probability mass.
            Filters tail tokens. Combined with temp=0.7 reduces hallucinations.
        repetition_penalty=1.1: Multiply prob of seen tokens by 1/1.1. Reduces
            loops like "the the the". >1.0 penalizes, <1.0 encourages repeat.
        pad_token_id=eos_token_id: If generation hits max_new_tokens without EOS,
            pad with EOS. Prevents "Setting pad_token_id to eos_token_id" warning.
        eos_token_id: Stop generation when this token appears. Usually </s>.

    Returns:
        str: Full text = prompt + generated response. Includes "### Response:".
             Slice after "### Response:\n" if you want answer only.
    """
    prompt = build_instruction_prompt(instruction, input_text)

    # Tokenize prompt to tensors.
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
     ).to(final_instruction_model.device) # Move to GPU/CPU matching model

    # Inference mode: no gradients, faster, less memory
    with torch.no_grad():
        # Generate tokens autoregressively until EOS or max_new_tokens
        outputs = final_instruction_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens, # Stop after N new tokens
            do_sample=True, # Enable sampling
            temperature=0.7, # Randomness control
            top_p=0.9, # Nucleus sampling threshold
            repetition_penalty=1.1, # Penalize repeated n-grams
            pad_token_id=tokenizer.eos_token_id, # Pad with EOS if needed
            eos_token_id=tokenizer.eos_token_id, # Stop token
        )

    # Decode tensor -> string. skip_special_tokens=True removes <s>, </s>
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [81]:
# ============================================================
# Test instruction-tuned pharma model
# ============================================================

test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?",
    "Summarize the role of lipid nanoparticles in mRNA vaccines.",
    "Why should AI predictions in drug discovery be experimentally validated?",
]

for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    print(generate_instruction_response(question, max_new_tokens=150))

[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin acts by blocking the conversion of glucose to glucolipotion in the liver. This results in a decrease in the blood glucose level and the formation of ketone bodies, which is what occurs in diabetes mellitus.

QUESTION:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why can atorvastatin and ezetimibe reduce LDL-C more effectively together?

### Response:
The combination of atorvastatin and ezetimibe has been shown to be superior in the treatment of patients with hypercholesterolemia than either drug alone. Atorvastatin, a HMG-CoA reductase inhibitor, lowers triglycerides and high-density lipoprotein cholesterol (HDL-C) and increases low-density lipoprotein cholesterol (LDL-C). Ezetimibe is an inhibitor of the JAK2 pathway that reduces intravascular lipids by increasing cholesterol excretion from the intestine and decreasing hepatic synthesis. These
QUESTION:
Summarize the role of lipid nanoparticles in mRNA vaccines.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Summarize the role of lipid nanoparticles in mRNA vaccines.

### Response:
- Lipid Nanoparticles are very small particles that can be used to deliver a gene into the body. These particles are created by a process called lipidation. A liposome is formed from a mixture of glycerol, cholesterol and other fatty acids. The liposomes are then packaged into a liposome capsule, which is then injected into the body. In order for these particles to penetrate the cell membrane they need to be unstable so they are surrounded by phospholipids. Liposomes are able to pass through the cells without causing damage because they have no water content. Liposomes contain genes, RNA or DNA,
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:
### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:

There are many reasons for why the drug discovery process needs to be validated.  The main reason i

In [82]:
# ============================================================
# Merge instruction-tuned LoRA adapter into base model
# ============================================================
# This creates a standalone instruction-tuned model.
# Later, we can use this merged model as the base model for preference tuning.

import os
import gc
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Path where the final merged instruction-tuned model will be saved.
merged_instruction_model_dir = "/content/pharma_tinyllama_instruction_merged_model"

os.makedirs(merged_instruction_model_dir, exist_ok=True)

In [83]:
# Load the original base model in normal precision for safe merging.
base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    merged_model_dir,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

# Load the tokenizer.
tokenizer_for_merge = AutoTokenizer.from_pretrained(
    config.model_name,
    trust_remote_code=True,
)

if tokenizer_for_merge.pad_token is None:
    tokenizer_for_merge.pad_token = tokenizer_for_merge.eos_token

# Attach the final instruction-tuned LoRA adapter.
model_with_instruction_adapter = PeftModel.from_pretrained(
    base_model_for_merge,
    instruction_adapter_dir,
)

# Merge LoRA adapter weights into the base model weights.
merged_instruction_model = model_with_instruction_adapter.merge_and_unload()

# Save the standalone merged model and tokenizer.
merged_instruction_model.save_pretrained(merged_instruction_model_dir)
tokenizer_for_merge.save_pretrained(merged_instruction_model_dir)

print(f"Merged instruction-tuned model saved to: {merged_instruction_model_dir}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged instruction-tuned model saved to: /content/pharma_tinyllama_instruction_merged_model


In [94]:
# Fix the base_model path in the merged model's README if it exists
import os

card_path = "/content/pharma_tinyllama_instruction_merged_model/README.md"

if os.path.exists(card_path):
    with open(card_path, "r") as f:
        content = f.read()

    content = content.replace(
        "base_model: /content/pharma_tinyllama_merged_model",
        "base_model: SivaSai8143/pharma-tinyllama-non-instruction-merged"
    )

    with open(card_path, "w") as f:
        f.write(content)
    print("Fixed base_model in README.md")

# Push merged model
push_to_hub(
    "SivaSai8143/pharma-tinyllama-instruction-merged", "model",
    local_path="/content/pharma_tinyllama_instruction_merged_model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...d_model/model.safetensors:   1%|          | 16.0MB / 2.20GB            

Pushed /content/pharma_tinyllama_instruction_merged_model -> SivaSai8143/pharma-tinyllama-instruction-merged (model)


| Fine-tuning stage         | Data format                 | What Model learns?              |
| ------------------------- | --------------------------- | ----------------------------------- |
| **Non-instruction FT**    | Raw text                    | Domain language or knowledge style |
| **Instruction FT**        | Instruction → Response      | Answers to user instruction     |
| **DPO Preference Tuning** | Prompt → Chosen vs Rejected | Prefers better answer          |


# Stage 3: Preference Tuning with DPO

In Stage 1, we adapted the model to the pharma domain using raw non-instruction text.

In Stage 2, we instruction-tuned the model using instruction-response data.

In Stage 3, we will use **preference data** with DPO.

DPO data has three main columns:

```text
prompt
chosen
rejected
```

- `prompt` is the user instruction.
- `chosen` is the preferred/better answer.
- `rejected` is the weaker answer.

The goal of DPO is to make the model prefer the `chosen` response over the `rejected` response.

Paper link: https://arxiv.org/pdf/2305.18290

| Section                       | Simple Meaning                                                                                         | Key Point                                                                           |
| ----------------------------- | ------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------- |
| **DPO Full Form**             | Direct Preference Optimization                                                                         | The model is trained directly using preference data.                                |
| **Paper**                     | “Direct Preference Optimization”                                                                       | Published at NeurIPS 2023 by Stanford researchers.                                  |
| **Main Idea**                 | Teach the model which answer is better and which answer is weaker.                                     | The model learns to prefer the `chosen` answer and avoid the `rejected` answer.     |
| **Before DPO: RLHF**          | RLHF usually has three stages.                                                                         | SFT → Reward Model → PPO                                                            |
| **RLHF Problem**              | RLHF is complex, expensive, and unstable.                                                              | Training a reward model and using PPO require high compute and careful tuning.      |
| **DPO Insight**               | A separate reward model is not required.                                                               | The language model itself can act like an implicit reward model.                    |
| **DPO Dataset Format**        | Each sample has three main fields.                                                                     | `prompt`, `chosen`, and `rejected`                                                  |
| **Prompt**                    | The user question or instruction.                                                                      | Example: “Explain the mechanism of metformin.”                                      |
| **Chosen**                    | The better or preferred answer.                                                                        | Usually accurate, complete, safe, and well-structured.                              |
| **Rejected**                  | The weaker or rejected answer.                                                                         | Usually vague, incomplete, incorrect, or unsafe.                                    |
| **DPO Training Goal**         | Increase the probability of the preferred answer.                                                      | The model becomes more likely to generate answers like the `chosen` response.       |
| **Role of Rejected Answer**   | Shows the model what type of answer to avoid.                                                          | The model reduces the probability of the `rejected` style answer.                   |
| **Reference Model**           | Usually the SFT model.                                                                                 | It prevents the DPO model from drifting too far from the original fine-tuned model. |
| **Policy Model**              | The model being trained during DPO.                                                                    | It learns to prefer the `chosen` answer over the `rejected` answer.                 |
| **Beta β**                    | A control parameter.                                                                                   | It controls how strongly the model moves away from the reference model.             |
| **DPO Loss**                  | A binary classification-style loss.                                                                    | It trains the model to make the `chosen` answer win over the `rejected` answer.     |
| **Reward Model Needed?**      | No.                                                                                                    | DPO removes the need for separate reward model training.                            |
| **PPO Needed?**               | No.                                                                                                    | DPO works more like supervised training instead of reinforcement learning.          |
| **Sampling During Training?** | No.                                                                                                    | DPO does not require an expensive generation loop like PPO.                         |
| **Main Advantage**            | Simpler and more stable.                                                                               | Easier to implement compared to traditional RLHF.                                   |
| **Training Cost**             | Lower than RLHF.                                                                                       | Only the policy model is trained.                                                   |
| **DPO vs SFT**                | SFT teaches the model how to answer.                                                                   | DPO teaches the model which answer is better.                                       |
| **DPO vs RLHF**               | RLHF uses a reward model and PPO.                                                                      | DPO directly uses preference loss.                                                  |
| **Gradient Intuition**        | Stronger updates happen when the model ranks answers incorrectly.                                      | The model learns more from difficult examples.                                      |
| **Practical Pipeline**        | Start with an SFT model, add preference data, then train with DPO.                                     | This creates a simple alignment pipeline.                                           |
| **Common Beta Value**         | The paper commonly used `β = 0.1`.                                                                     | For summarization tasks, `β = 0.5` was also used.                                   |
| **Experiments**               | Tested on sentiment, summarization, and dialogue tasks.                                                | DPO can perform equal to or better than PPO.                                        |
| **Limitation**                | Large-scale training, reward hacking, and out-of-distribution generalization are still open questions. | DPO is powerful, but not perfect.                                                   |
| **One-Liner**       | DPO teaches the model which answer is better.                                                          | Instruction tuning teaches answering; DPO teaches preference.                       |


In [84]:
# ============================================================
# 25. Install TRL for DPO training
# ============================================================
# Install TRL (Transformer Reinforcement Learning) library from Hugging Face.
# TRL provides DPOTrainer and DPOConfig for preference tuning.

!pip install -q -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 44.5 MB/s eta 0:00:00


In [85]:
# ============================================================
# 26. Load DPO preference dataset
# ============================================================
# Expected columns: prompt, chosen, rejected
# File format: JSONL where each line is one JSON object:
# {"prompt": "...", "chosen": "...", "rejected": "..."}

from datasets import load_dataset

preference_data_path = "/content/pharma_preference_dataset.jsonl"

# Load JSONL file into HuggingFace Dataset object
preference_dataset = load_dataset(
    "json", # Parse as JSON lines
    data_files=preference_data_path, # Path to.jsonl file
    split="train" # Treat whole file as train split
)

print(preference_dataset)
print(preference_dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
    num_rows: 48
})
{'prompt': '### Instruction:\nExplain the primary mechanism of action of metformin.\n\n### Response:\n', 'chosen': 'Metformin primarily acts by activating AMP-activated protein kinase, also called AMPK. AMPK is a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while reducing hepatic gluconeogenesis, which helps lower blood glucose levels.', 'rejected': 'Metformin mainly works by increasing insulin secretion from the pancreas, and kidney function is usually not very relevant. Its side effects are generally not important unless the patient feels very sick.', 'source_page': 1, 'topic': 'Metformin pharmacology'}


In [86]:
# Create train-validation split
preference_dataset = preference_dataset.train_test_split(
    test_size=0.15,
    seed=42
)

# Rename test split to validation split
preference_dataset["validation"] = preference_dataset.pop("test")

print("After train-validation split:")
print(preference_dataset)
print("Train rows:", len(preference_dataset["train"]))
print("Validation rows:", len(preference_dataset["validation"]))

After train-validation split:
DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 40
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'source_page', 'topic'],
        num_rows: 8
    })
})
Train rows: 40
Validation rows: 8


## Preference Tuning Base Model

For DPO, we use the **merged instruction-tuned model** as the base model.

Then we attach a **new LoRA adapter** for preference tuning.

This gives us the flow:

```text
Merged instruction-tuned model
        +
New preference LoRA adapter
        ↓
DPO preference tuning
```


In [87]:
merged_instruction_model_dir

'/content/pharma_tinyllama_instruction_merged_model'

In [88]:
# ============================================================
# Load merged instruction model as base for preference tuning
# ============================================================

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

use_cuda = torch.cuda.is_available()

if use_cuda:
    # GPU: Load merged SFT model in 4-bit NF4. Saves VRAM for policy+ref model
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir, # Stage1+Stage2 merged weights
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4", # Best quality 4-bit format
            bnb_4bit_compute_dtype=torch.float16, # Compute in fp16
            bnb_4bit_use_double_quant=True, # Extra 0.4bit compression
        ),
        device_map="auto", # Shard across GPUs
        trust_remote_code=True,
    )

    # Prep for QLoRA: cast layernorms to fp32, set requires_grad for inputs
    preference_base_model = prepare_model_for_kbit_training(preference_base_model)

else:
    # CPU: Load in fp32. DPO on CPU is very slow, only for debug
    preference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable KV cache: incompatible with gradient checkpointing during training
preference_base_model.config.use_cache = False

# Create a new LoRA adapter for preference tuning.
preference_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # Causal LM for text generation
    r=16, # Rank: same as Stage-1/2 for consistency
    lora_alpha=32, # Scaling: alpha/r = 32/16 = 2.0x
    lora_dropout=0.05, # Dropout on LoRA layers
    bias="none", # Don't train bias terms
    target_modules=[ # Llama attention + MLP layers
        "q_proj", "k_proj", "v_proj", "o_proj", # Attention
        "gate_proj", "up_proj", "down_proj", # SwiGLU MLP(Multi-Layer Perceptron)
    ],
)

# Wrap base with new trainable LoRA adapter
preference_model = get_peft_model(
    preference_base_model,
    preference_lora_config,
)

preference_model.print_trainable_parameters()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


In [89]:
# Define directories for Stage-3 Direct Preference Optimization artifacts.

import os
from trl import DPOTrainer
from trl import DPOConfig

preference_output_dir = "/content/pharma_tinyllama_preference_dpo_output"
preference_adapter_dir = "/content/pharma_tinyllama_preference_dpo_lora_adapter"

# Create dirs if missing. exist_ok=True prevents FileExistsError
os.makedirs(preference_output_dir, exist_ok=True)
os.makedirs(preference_adapter_dir, exist_ok=True)

In [90]:
# ============================================================
# 30. Create DPO training arguments
# ============================================================
# Configure Direct Preference Optimization hyperparameters via DPOConfig.

from trl import DPOConfig

dpo_training_args = DPOConfig(
    output_dir=preference_output_dir, # Checkpoints + logs go here

    # Training duration
    num_train_epochs=3, # Full passes over dataset
    max_steps=5, # Override for debug, remove in prod

    # Batch settings
    per_device_train_batch_size=1, # DPO processes chosen+rejected per sample
    per_device_eval_batch_size=1, # Eval batch size
    gradient_accumulation_steps=8, # Effective batch = 1*8 = 8 pairs

    # Optimizer settings
    learning_rate=5e-5, # Lower than SFT: 5e-5 vs 1e-4
    warmup_steps=2, # Short warmup for DPO
    weight_decay=0.01, # L2 regularization

    # Logging and evaluation
    logging_steps=1, # Log loss every step
    logging_first_step=True, # Log step 0
    eval_strategy="steps", # Eval every eval_steps
    eval_steps=1, # Frequent eval for debug

    # Checkpoint saving
    save_steps=5, # Save every 5 steps
    save_total_limit=2, # Keep only last 2 ckpts

    # Precision settings
    fp16=False, # Don't set with 4-bit base
    bf16=False, # Use compute_dtype in BitsAndBytesConfig instead

    # Disable external logging tools
    report_to="none", # No wandb

    # Keep required columns
    remove_unused_columns=False, # Critical: DPO needs prompt/chosen/rejected

    # DPO hyperparameter
    beta=0.1, # KL penalty: lower = more deviation from SFT
)

print(dpo_training_args)

DPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.1,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_num_proc=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_dropout=True,
disable_tqdm=False,
discopop_tau=0.05,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat

In [91]:
# ============================================================
# 31. Build DPOTrainer
# ============================================================

from trl import DPOTrainer

dpo_trainer = DPOTrainer(
    model=preference_model,
    ref_model=None,  # None means TRL will internally use the reference behavior
    args=dpo_training_args,
    train_dataset=preference_dataset["train"],
    eval_dataset=preference_dataset["validation"],
    processing_class=tokenizer,
)

print("DPOTrainer is ready.")

Adding EOS to train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/40 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

DPOTrainer is ready.


In [92]:
# ============================================================
# 32. Start DPO preference tuning
# ============================================================

dpo_train_result = dpo_trainer.train()

print("DPO preference tuning completed.")
print(dpo_train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.693147,0.693147,2.127841,1413.000000,-3.536075,-3.539604,0.553208,0.000000,0.000000,0.000000,0.000000,-127.274981,-120.277911
2,0.693147,0.650178,2.125521,2789.000000,-3.537148,-3.541861,0.555073,0.023541,-0.065130,1.000000,0.088672,-127.039569,-120.929215
3,0.660598,0.532647,2.120212,4051.000000,-3.538658,-3.545292,0.559571,0.070376,-0.289814,1.000000,0.360190,-126.571218,-123.176053
4,0.546336,0.464975,2.115262,5380.000000,-3.539335,-3.547155,0.559571,0.101142,-0.440878,1.000000,0.542020,-126.263557,-124.686689
5,0.470978,0.434318,2.112991,6698.000000,-3.539675,-3.548242,0.559571,0.115192,-0.515710,1.000000,0.630902,-126.123059,-125.435010


DPO preference tuning completed.
TrainOutput(global_step=5, training_loss=0.6128411412239074, metrics={'train_runtime': 77.4655, 'train_samples_per_second': 0.516, 'train_steps_per_second': 0.065, 'total_flos': 49278084096000.0, 'train_loss': 0.6128411412239074, 'epoch': 1.0})


| Parameter               | Short Meaning                                                                              |
| ----------------------- | ------------------------------------------------------------------------------------------ |
| **Step**                | Current optimizer step during training.                                                    |
| **Training Loss**       | DPO loss on the training data; lower is generally better.                                  |
| **Validation Loss**     | DPO loss on unseen validation data; helps check generalization.                            |
| **Entropy**             | Measures how uncertain the model is; higher means more random, lower means more confident. |
| **Num Tokens**          | Total number of tokens processed so far.                                                   |
| **Logits/chosen**       | Raw model score for the preferred answer.                                                  |
| **Logits/rejected**     | Raw model score for the rejected answer.                                                   |
| **Mean Token Accuracy** | Average token-level prediction accuracy.                                                   |
| **Rewards/chosen**      | DPO implicit reward for the preferred answer; should be higher.                            |
| **Rewards/rejected**    | DPO implicit reward for the rejected answer; should be lower.                              |
| **Rewards/accuracies**  | How often the model ranks the chosen answer above the rejected answer.                     |
| **Rewards/margins**     | Difference between chosen reward and rejected reward; positive is good.                    |
| **Logps/chosen**        | Log probability of the chosen answer; less negative means more likely.                     |
| **Logps/rejected**      | Log probability of the rejected answer; ideally more negative than chosen.                 |


Simple summary: In DPO training, the main goal is to make the model assign higher probability and higher reward to the chosen answer than the rejected answer.

In [93]:
# ============================================================
# 33. Save DPO preference-tuned LoRA adapter
# ============================================================

# Save only the Stage-3 DPO LoRA weights + config to disk. Base model stays unchanged.
# Writes adapter_config.json and adapter_model.safetensors
# This adapter contains preference tuning deltas on top of merged instruction model.
dpo_trainer.model.save_pretrained(preference_adapter_dir)
# Save tokenizer files alongside adapter to ensure identical tokenization at inference.
tokenizer.save_pretrained(preference_adapter_dir)

print(f"Preference-tuned LoRA adapter saved to: {preference_adapter_dir}")
# List files to verify save succeeded.
print(os.listdir(preference_adapter_dir))


Preference-tuned LoRA adapter saved to: /content/pharma_tinyllama_preference_dpo_lora_adapter
['ref', 'README.md', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json']


In [94]:
# ============================================================
# Push Stage 3 DPO LoRA adapter to Hugging Face
# ============================================================

# Stage 3: DPO preference LoRA adapter
HF_REPO_DPO_ADAPTER = f"SivaSai8143/pharma-tinyllama-dpo-lora-adapter"

# Stage 3 final merged model
HF_REPO_DPO_MERGED = f"SivaSai8143/pharma-tinyllama-dpo-merged"

dpo_trainer.model.push_to_hub(
    HF_REPO_DPO_ADAPTER,
    private=False
)

tokenizer.push_to_hub(
    HF_REPO_DPO_ADAPTER,
    private=False
)

print("Stage 3 DPO LoRA adapter pushed to:")
print(HF_REPO_DPO_ADAPTER)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

  ...adapter_model.safetensors:   1%|1         |  348kB / 25.3MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 3 DPO LoRA adapter pushed to:
SivaSai8143/pharma-tinyllama-dpo-lora-adapter


In [95]:
# ============================================================
# 34. Reload preference-tuned model for inference
# ============================================================

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Clear Python garbage collector to free RAM before loading large models
gc.collect()

# Clear CUDA cache to release GPU memory held by previous training objects
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if use_cuda:
    # GPU: Reload the merged Stage1+Stage2 base in 4-bit to save VRAM
    # This is the same merged_instruction_model_dir used during DPO training
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, # 4-bit NF4 quantization
            bnb_4bit_quant_type="nf4", # Normalized float 4-bit format
            bnb_4bit_compute_dtype=torch.float16, # Compute in fp16 for speed
            bnb_4bit_use_double_quant=True, # Second quantization for 0.4bit savings
        ),
        device_map="auto", # Auto-shard across available GPUs
        trust_remote_code=True, # Allow custom model code if needed
    )
else:
    # CPU: Load base in fp32. Slower inference, use only for debug
    preference_inference_base_model = AutoModelForCausalLM.from_pretrained(
        merged_instruction_model_dir,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Load Stage-3 DPO LoRA adapter on top of merged instruction base
# This stacks: Base TinyLlama + Stage1 Domain + Stage2 Instruction + Stage3 DPO
preference_inference_model = PeftModel.from_pretrained(
    preference_inference_base_model, # Frozen base with domain+instruction merged
    preference_adapter_dir, # DPO LoRA adapter from previous save
)


# Set to eval mode: disables dropout, fixes batchnorm stats
# Required for deterministic inference, avoids training-time randomness
# ------------------------
# You need .eval() because PyTorch models have layers that behave differently in training vs inference.
# preference_inference_model.eval()  # Remove this line

# output1 = model.generate(prompt_ids)  # "Warfarin requires INR..."
# output2 = model.generate(prompt_ids)  # "Warfarin needs monitoring..."
# ------------------------
preference_inference_model.eval()

print("Preference-tuned model loaded successfully for inference.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Preference-tuned model loaded successfully for inference.


In [96]:
# ============================================================
# 35. Preference-tuned inference helper
# ============================================================

def build_preference_prompt(instruction, input_text=""):
    # Remove leading/trailing whitespace to avoid token drift vs training format
    instruction = instruction.strip()
    input_text = input_text.strip()

    # Build prompt using exact template from SFT/DPO training to match tokenization
    if input_text:
        return (
            f"### Instruction:\n{instruction}\n\n"  # Alpaca-style instruction block
            f"### Input:\n{input_text}\n\n"        # Optional context field
            f"### Response:\n"                     # Model generates after this tag
        )

    # No-input variant: instruction only, matches training samples with empty input
    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Response:\n"
    )

In [97]:
def generate_preference_response(instruction, input_text="", max_new_tokens=150):
    # Build prompt with exact template used in SFT/DPO training
    prompt = build_preference_prompt(instruction, input_text)

    # Tokenize prompt and move tensors to same device as model for GPU inference
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(preference_inference_model.device)

    # Disable gradient tracking for inference to save memory and speed up
    with torch.no_grad():
        outputs = preference_inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens, # Cap generation length
            do_sample=True, # Enable sampling vs greedy decoding
            temperature=0.7, # Lower = more deterministic, 0.7 balances creativity/coherence
            top_p=0.9, # Nucleus sampling: pick from top 90% probability mass
            repetition_penalty=1.1, # Penalize repeat tokens to reduce loops
            pad_token_id=tokenizer.eos_token_id, # Use EOS as pad to avoid undefined pad token
            eos_token_id=tokenizer.eos_token_id, # Stop generation at EOS
        )

    # Decode token IDs back to string, strip special tokens like <s>, </s>
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [98]:
# ============================================================
# 36. Test preference-tuned pharma model
# ============================================================

# Sample domain questions to sanity-check DPO model behavior post-training
preference_test_questions = [
    "Explain the primary mechanism of action of metformin.",
    "Why should AI predictions in drug discovery be experimentally validated?",
    "Define pharmacovigilance.",
    "Explain why pharmacovigilance continues after drug approval.",
]

# Run inference on each test question and print prompt + response
for question in preference_test_questions:
    print("=" * 100) # Separator for readability
    print("QUESTION:")
    print(question)

    print("\nMODEL RESPONSE:")
    # Generate with DPO-tuned model using default sampling params: temp=0.7, top_p=0.9
    print(generate_preference_response(question, max_new_tokens=150))


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
Explain the primary mechanism of action of metformin.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Explain the primary mechanism of action of metformin.

### Response:
Metformin has been shown to inhibit glucose uptake into cells and to reduce hepatic gluconeogenesis by blocking the activity of the enzymes gluconeogenic flux and gluconeogenic capacity. It is also able to inhibit the activity of the AMP-activated protein kinase, which plays a role in insulin secretion from pancreatic beta cells. The mechanism for the reduction of hepatic gluconeogenesis and hyperglycemia are not completely understood but may involve increased clearance of glucose from the liver and decreased production of fatty acids via gluconeogenesis. Metform
QUESTION:
Why should AI predictions in drug discovery be experimentally validated?

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Why should AI predictions in drug discovery be experimentally validated?

### Response:
The purpose of the study is to examine whether there are differences between the predicted AI and experimental data. In order to do this, a new dataset needs to be created from the drug discovery literature. 

The drugs will be categorized into four groups based on their mechanism of action (MOA). The four groups will be:
- Compounds with unknown MOA (Group 1)
- Moderately known MOAs (Group 2)
- Known MOAs (Group 3)
- Fully known MOAs (Group 4)

We have developed a new methodology for classification called 3-class classification where we predict if the compound has been classified as Group 1
QUESTION:
Define pharmacovigilance.

MODEL RESPONSE:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Define pharmacovigilance.

### Response:
The pharmacovigilance is the systematic study of safety of drugs, devices and other health-related products in order to identify, monitor and prevent adverse events and incidents (safety issues) that may occur during their development, testing, marketing and use. Pharmacovigilance studies are an important tool for regulatory agencies like FDA and EMA to assess risks associated with new drugs. 
Pharmacovigilance also assists the FDA in ensuring the public is protected by monitoring the side effects of drugs. The FDA's mission is to protect the public health by assuring the safety, efficacy, and security of human and
QUESTION:
Explain why pharmacovigilance continues after drug approval.

MODEL RESPONSE:
### Instruction:
Explain why pharmacovigilance continues after drug approval.

### Response:
Pharmacovigilance is an important aspect of the entire drug development process, and its goal is to ensure that a drug does not cause any 

In [99]:
# ============================================================
# 37. Optional: Merge DPO preference adapter into the instruction-tuned base model
# ============================================================
# Use this only after preference tuning is complete and you want a standalone final model.
# Merges LoRA weights into base for single-file deployment. Trade-off: lose ability to swap/remove DPO later.

import os
import gc
import torch

from transformers import AutoModelForCausalLM
from peft import PeftModel

# Force Python garbage collection to release RAM before heavy merge op
gc.collect()

# Clear GPU cache to free VRAM held by inference model from previous cell
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Define output path for final merged model: Stage1+Stage2+Stage3 baked into one checkpoint
final_merged_preference_model_dir = "/content/pharma_tinyllama_final_preference_merged_model"
# Create directory if missing, prevents FileNotFoundError on save
os.makedirs(final_merged_preference_model_dir, exist_ok=True)

In [100]:
# Load the merged instruction model in normal precision for safe merging.
base_model_for_preference_merge = AutoModelForCausalLM.from_pretrained(
    merged_instruction_model_dir, # Stage1+Stage2 merged base from earlier
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, # fp16 on GPU, fp32 on CPU
    device_map="auto" if torch.cuda.is_available() else None, # Shard across GPUs if available
    trust_remote_code=True, # Allow custom TinyLlama code
)

# Attach the DPO preference LoRA adapter.
model_with_preference_adapter = PeftModel.from_pretrained(
    base_model_for_preference_merge, # Frozen instruction-tuned base
    preference_adapter_dir, # DPO LoRA delta weights
)

# Merge the preference adapter into the instruction-tuned base model.
final_merged_preference_model = model_with_preference_adapter.merge_and_unload()

# Save final standalone model and tokenizer.
final_merged_preference_model.save_pretrained(final_merged_preference_model_dir)
# Save tokenizer with model so inference uses identical vocab/special tokens
tokenizer.save_pretrained(final_merged_preference_model_dir)

print(f"Final merged preference-tuned model saved to: {final_merged_preference_model_dir}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final merged preference-tuned model saved to: /content/pharma_tinyllama_final_preference_merged_model


In [101]:
# ============================================================
# Push Stage 3 Merged DPO Model to Hugging Face
# ============================================================
# Stage 3 final merged model
HF_REPO_DPO_MERGED = f"SivaSai8143/pharma-tinyllama-dpo-merged"

final_merged_preference_model.model.push_to_hub(
    HF_REPO_DPO_MERGED,
    private=False
)

tokenizer.push_to_hub(
    HF_REPO_DPO_MERGED,
    private=False
)

print("Stage 3 Merged DPO Model pushed to:")
print(HF_REPO_DPO_MERGED)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...wwizcr5/model.safetensors:   1%|          | 16.0MB / 2.07GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 3 Merged DPO Model pushed to:
SivaSai8143/pharma-tinyllama-dpo-merged
